[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tulin206/TUD_CNN_2026/blob/main/code/CNN/10_vgg_cifar10_pytorch_exercise.ipynb)


# CIFAR-10 classification with a VGG-like network in PyTorch — Exercise

Build a CNN from scratch to classify 32×32 RGB images into 10 classes. Work through data loading, model construction, training, learning curves, test evaluation, and prediction visualization.

Compare with the [TensorFlow/Keras notebook](10_vgg_cifar10_exercise.ipynb). This is a simplified teaching architecture, not a pretrained model.


## Your task

Complete the TODO cells for the architecture, optimizer and loss, training loop, learning curves, and evaluation. Data loading and the evaluation helper are provided. Each unfinished cell raises a clear error until completed; run cells in order after filling them in.

Try the exercise before consulting the [PyTorch solution](10_vgg_cifar10_pytorch.ipynb).


## Setup

In Colab, upload this notebook and select a GPU runtime if available. Locally, use JupyterLab with a Python environment containing PyTorch and torchvision. If dependencies are missing, run:

```python
%pip install torch torchvision numpy matplotlib pandas seaborn scikit-learn
```

The initial CIFAR-10 download requires internet access. Data is cached in `./data`. CPU training works but can be slow. Start with `EPOCHS = 1` to check the workflow, then use the full epoch count. CUDA, Apple MPS, or CPU is selected automatically.


In [ ]:
import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print('Device:', device)
# Seeds help repeat experiments; different devices may still produce different results.
BATCH_SIZE = 64
LEARNING_RATE = 0.001

EPOCHS = 30


## TensorFlow/Keras to PyTorch

| TensorFlow/Keras | PyTorch in this notebook |
| --- | --- |
| Images shaped `(N, H, W, C)` | Images shaped `(N, C, H, W)` |
| `Conv2D(..., padding='same')` with 3×3 kernels and stride 1 | `nn.Conv2d(..., kernel_size=3, padding=1)` |
| `Dense` | `nn.Linear` (input size must be specified) |
| Softmax output + sparse categorical cross-entropy | Raw logits + `nn.CrossEntropyLoss()`; labels are integer class indices |
| `model.compile()` and `model.fit()` | Explicit loss, optimizer, and training loop |
| `model.predict()` | `model.eval()` and `torch.no_grad()` around a forward pass |

`CrossEntropyLoss` already includes the log-softmax calculation. Do not apply softmax before this loss. `argmax(dim=1)` selects the predicted class; use softmax only when you want probabilities.

These models match the layer widths and pooling pattern of the corresponding Keras examples, but weights and numerical results will differ. Here we split the 50,000 training images into 45,000 training and 5,000 validation images; the original Keras examples use all 50,000 for training and reuse the test set for validation. Use identical splits and settings for a controlled framework comparison.


## Load and inspect CIFAR-10

Use `Dataset` objects to access images and labels and `DataLoader` objects to form batches. Only the training loader shuffles its samples.


In [ ]:
# ToTensor converts PIL images to float32 tensors, scales pixels to [0, 1],
# and changes the shape from (H, W, C) to (C, H, W).
transform = transforms.ToTensor()
full_train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
class_names = full_train_dataset.classes

# Keep the test set out of training and validation.
train_dataset, val_dataset = random_split(
    full_train_dataset, [45000, 5000], generator=torch.Generator().manual_seed(SEED)
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          generator=torch.Generator().manual_seed(SEED), num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print('Training / validation / test:', len(train_dataset), len(val_dataset), len(test_dataset))
images, labels = next(iter(train_loader))
print('Image batch:', images.shape, images.dtype)
print('Label batch:', labels.shape, labels.dtype)


In [ ]:
plt.figure(figsize=(10, 10))
for i in range(25):
    image, label = train_dataset[i]
    plt.subplot(5, 5, i + 1)
    plt.imshow(image.permute(1, 2, 0))  # Matplotlib expects (H, W, C).
    plt.title(class_names[label])
    plt.axis('off')
plt.tight_layout()
plt.show()


## Build the VGG-like network

Three pooling layers reduce the spatial size from 32 to 16 to 8 to 4. The flattened feature vector therefore has 256 × 4 × 4 entries. `forward()` defines how input flows through the model.


In [ ]:
class VGGLike(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # TODO: Define self.features with three blocks:
        # Conv(3->64), Conv(64->64), Pool
        # Conv(64->128), Conv(128->128), Pool
        # Conv(128->256), Conv(256->256), Pool
        # Use 3x3 kernels, padding=1, ReLU after every convolution, and 2x2 pooling.
        # TODO: Define self.classifier:
        # Flatten -> Linear(256*4*4, 512) -> ReLU -> Dropout(0.5) -> Linear(512, 10)
        raise NotImplementedError('Complete the VGG architecture before continuing.')

    def forward(self, x):
        # TODO: Apply the feature extractor, then the classifier. Return logits.
        raise NotImplementedError('Implement forward().')


In [ ]:
model = VGGLike().to(device)
print(model)
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))
model.eval()
with torch.no_grad():
    output = model(torch.zeros(2, 3, 32, 32, device=device))
assert output.shape == (2, 10)
print('Output shape:', output.shape)


## Loss and optimizer

Use Adam with learning rate 0.001 and cross-entropy for integer labels. PyTorch optimizers receive the model parameters explicitly.


In [ ]:
# TODO: Define criterion = nn.CrossEntropyLoss() and an Adam optimizer using model.parameters().
raise NotImplementedError('Define criterion and optimizer.')


## Training and validation

Training requires clearing gradients, a forward pass, backpropagation, and a parameter update. Validation uses evaluation mode and disables gradient tracking. Losses are weighted by batch size so the smaller final batch is counted correctly.


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    # TODO: Switch to training mode and initialize loss, correct, and total counters.
    # For each batch: move tensors to device, zero gradients, compute logits/loss,
    # call loss.backward(), then optimizer.step().
    # Accumulate loss.item() * batch_size and the number of correct predictions.
    # Return sample-weighted mean loss and accuracy.
    raise NotImplementedError('Implement one training epoch.')


In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()  # Disables dropout; no parameter updates during evaluation.
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total


In [ ]:
history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}
for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    history['loss'].append(train_loss)
    history['accuracy'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_acc)
    print(f'Epoch {epoch + 1}/{EPOCHS} | loss={train_loss:.4f} accuracy={train_acc:.3f} '
          f'| val_loss={val_loss:.4f} val_accuracy={val_acc:.3f}')


## Learning curves

Compare training and validation loss and accuracy. If training improves while validation loss grows, what might that indicate?


In [ ]:
# TODO: Convert history to a DataFrame and plot training/validation accuracy and loss.
raise NotImplementedError('Plot the learning curves.')


## Final test evaluation

Evaluate on the held-out test set after choosing your model using validation results. Inspect which classes are confused most often.


In [ ]:
# TODO: Call evaluate(model, test_loader, criterion, device) and print loss and accuracy.
raise NotImplementedError('Evaluate the test set.')


In [ ]:
# TODO: In eval mode and torch.no_grad(), collect true labels and argmax predictions.
# Move predictions to CPU, then plot a confusion matrix with all 10 class names.
raise NotImplementedError('Create the confusion matrix.')


## Inspect individual predictions


In [ ]:
images, labels = next(iter(test_loader))
model.eval()
with torch.no_grad():
    logits = model(images.to(device))
    probabilities = torch.softmax(logits, dim=1)  # Only needed for probabilities.
    confidence, predictions = probabilities.max(dim=1)
predictions, confidence = predictions.cpu(), confidence.cpu()
plt.figure(figsize=(15, 6))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(images[i].permute(1, 2, 0))
    plt.title(f'True: {class_names[labels[i].item()]}\n'
              f'Pred: {class_names[predictions[i].item()]} ({confidence[i].item():.0%})')
    plt.axis('off')
plt.tight_layout()
plt.show()


## Reflect and experiment

1. Trace the shape after each convolution, pool, and flatten operation.
2. Why do we clear gradients at the start of each batch?
3. Why do validation and prediction use both `model.eval()` and `torch.no_grad()`?
4. Change one setting at a time (learning rate, filter count, or epoch count). Compare validation curves and training time.
5. Compare the Keras and PyTorch implementations. Which steps does Keras perform for you?

References: [PyTorch training and optimization](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html), [torchvision CIFAR10](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.CIFAR10.html).
